# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa - Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json) using the `mlcroissant` library. The dataset uses the Croissant standard for describing datasets in a machine-readable way, and records are referenced by their `@id` fields (as per Croissant best practice).

### Dataset Source
The dataset Croissant JSON-LD schema is available at:

[https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)


In [ ]:
# Ensure `mlcroissant` and dependencies are installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Croissant `metadata` is an object, not a dict; access fields with dot notation
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and associated `@id` values.

We will enumerate all record sets, fields, and columns by their `@id` values, which uniquely identify each entity in the dataset.

In [ ]:
# List available record sets and their fields (all referenced by their @id)
record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")

for recset in record_sets:
    print(f"RecordSet @id: {recset['@id']}")
    print(f"  Name: {recset.get('name', '(no name)')}")
    fields = recset.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields and Columns:")
    for f in fields:
        print(f"    Field @id: {f['@id']}, name: {f.get('name', '(no name)')}")
        if 'column' in f:
            columns = f['column']
            if isinstance(columns, dict):
                columns = [columns]
            for c in columns:
                print(f"      Column @id: {c['@id']} (name: {c.get('name', '(no name)')})")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. In line with Croissant, we use the record set and field `@id` identifiers shown above.

**Note:** For this dataset, there is typically one main record set containing survey records. We'll extract all data from the main record set.

In [ ]:
# Collect all record set @id values
rs_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
print("Available record_set @ids:\n", rs_ids)

# For this notebook, we use the first (main) record set.
main_record_set_id = rs_ids[0]

# Extract all records for each record set into a DataFrame
dataframes = {}
for rs_id in rs_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Display the available columns (which may include columns named by their @id or user-friendly name)
print(f"Columns in main record set ({main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's perform common EDA steps on the main record set: 
- Filtering records based on a numeric field
- Normalizing the field
- Grouping records by a categorical field

Below, we choose one numeric field and one grouping field using their `@id` from the schema overview table above.

In [ ]:
# Replace these @ids with actual ones from previous output.
# For demonstration, we assume the following field @ids (check overview cell output):
numeric_field_id = 'gad7_total_score'   # Example: GAD-7 assessment total score
group_field_id = 'level_of_education'   # Example: Education level for grouping

# If columns are stored as field @id, use those; otherwise adjust
df = dataframes[main_record_set_id]

if numeric_field_id not in df.columns:
    # Find a suitable numeric field
    numeric_candidates = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
    print('Numeric candidates:', numeric_candidates)
    numeric_field_id = numeric_candidates[0]

threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (N={len(filtered_df)}):")
print(filtered_df.head())

# Normalization
filtered_df[numeric_field_id + '_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

if group_field_id not in df.columns:
    # Fallback: Use the first string/categorical column
    cat_candidates = [col for col in df.columns if df[col].dtype == object]
    print('Group field candidates:', cat_candidates)
    group_field_id = cat_candidates[0]

grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
print(grouped_df.head())

## 5. Visualization
Visualize the chosen numeric field and its distribution, as well as the mean per group.

Here we'll plot a histogram/distribution of the numeric field and a bar chart of means per group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field (all records)
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.show()

# Bar plot of mean by group (if group is not too high cardinality)
if grouped_df.index.nunique() <= 20:
    plt.figure(figsize=(10,5))
    grouped_df.sort_values().plot(kind='bar')
    plt.ylabel(f'Mean {numeric_field_id}')
    plt.title(f'Mean {numeric_field_id} by {group_field_id}')
    plt.show()

## 6. Conclusion
In this notebook, we've:

- Loaded metadata and records using the Croissant schema via the `mlcroissant` library
- Enumerated the available record sets and their fields using `@id` references
- Loaded records into DataFrames for analysis
- Filtered and normalized numeric fields and grouped results for basic exploratory data analysis
- Visualized key data distributions and group effects

This structured approach, using the Croissant standard and referencing all schema elements by their `@id`, allows for reliable, reproducible, and standards-compliant analysis workflows. For more advanced work, refer to the [mlcroissant documentation](https://mlcommons.github.io/croissant/) to access further features such as data validation, transformations, and working with more complex schemas.
